# Directional Admission Budget Layer Test

This notebook checks the new budget-only admission behavior:

- budget is per `(source, target, slice)`, not just per source;
- negative bias creates budget with `ceil(strength * source_ues)`;
- target/source traffic safety values do not veto admission;
- `admit_candidates()` uses temporary counts only;
- `commit()` is the only real budget-consumption point;
- offset neutralization is exact-direction only;
- a real upper-env step exposes the same directional budget state.

In [ ]:
from pathlib import Path
import sys
from types import SimpleNamespace

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "safe_admission_controller.py").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from safe_admission_controller import DirectionalAdmissionBudgetController, SafeAdmissionController
from global_ppo_3gnb_env import GlobalPPO3GNBEnv
from upper_agent_training_scenarios import CENTER_GAP_GNB_CONFIGS

print(f"repo root: {ROOT}")
print(f"SafeAdmissionController alias: {SafeAdmissionController.__name__}")

## 1. Directional Budget Formula

With 6 eMBB UEs on gNB1:

- `bias[1 -> 0, eMBB] = -0.5` gives `ceil(0.5 * 6) = 3`;
- `bias[1 -> 2, eMBB] = -0.25` gives `ceil(0.25 * 6) = 2`.

In [ ]:
controller = DirectionalAdmissionBudgetController(bias_deadband=0.05)
controller.begin_upper_window(
    directional_bias=np.asarray([[[-0.5], [-0.25]]], dtype=float),
    neighbor_graph={1: [0, 2]},
    gnb_ids=[1],
    slice_types=["eMBB"],
    loads={(1, "eMBB"): 0.90},
    ue_counts={(1, "eMBB"): 6},
)
state = controller.get_state()
direction_quota = state["direction_quota"]

assert direction_quota[(1, 0, "eMBB")] == 3
assert direction_quota[(1, 2, "eMBB")] == 2
assert state["quota"][(1, "eMBB")] == 5

direction_quota

## 2. No Traffic-Safety Vetoes

This deliberately gives the candidate terrible target-load and target-SLA values.
The candidate is still accepted because it has directional budget. The second
candidate is rejected only because its direction has no budget.

In [ ]:
controller = DirectionalAdmissionBudgetController(bias_deadband=0.05)
controller.begin_upper_window(
    directional_bias=np.asarray([[[-1.0], [0.5]]], dtype=float),
    neighbor_graph={0: [1, 2]},
    gnb_ids=[0],
    slice_types=["eMBB"],
    loads={(0, "eMBB"): 0.10, (1, "eMBB"): 2.00},
    ue_counts={(0, "eMBB"): 3},
)

accepted, rejected, debug = controller.admit_candidates([
    {
        "ue_id": 10,
        "source_id": 0,
        "target_id": 1,
        "slice_type": "eMBB",
        "a3_margin": 5.0,
        "target_load": 10.0,
        "target_total_load": 10.0,
        "target_load_increment": 10.0,
        "target_safe_limit": 0.1,
        "target_total_safe_limit": 0.1,
        "target_sla_severity": 1.0,
    },
    {
        "ue_id": 11,
        "source_id": 0,
        "target_id": 2,
        "slice_type": "eMBB",
        "a3_margin": 4.0,
    },
])

state = controller.get_state()
assert [item["ue_id"] for item in accepted] == [10]
assert rejected[0]["rejection_reason"] == "no_directional_budget"
assert state["stats"]["rejected_target_safety"] == 0
assert state["stats"]["rejected_target_total_safety"] == 0
assert state["stats"]["rejected_target_sla"] == 0
assert state["stats"]["rejected_no_source_excess"] == 0

{
    "accepted": [(item["ue_id"], item["target_id"]) for item in accepted],
    "rejected": [(item["ue_id"], item["rejection_reason"]) for item in rejected],
    "stats": state["stats"],
}

## 3. Temporary Selection vs Real Commit

`admit_candidates()` should not consume real budget. It uses temporary counts
inside the batch, then `commit()` consumes exactly one slot after a successful
handover.

In [ ]:
controller = DirectionalAdmissionBudgetController(bias_deadband=0.05)
controller.begin_upper_window(
    directional_bias=np.asarray([[[-0.5]]], dtype=float),
    neighbor_graph={0: [1]},
    gnb_ids=[0],
    slice_types=["eMBB"],
    ue_counts={(0, "eMBB"): 4},
)

candidates = [
    {"ue_id": ue_id, "source_id": 0, "target_id": 1, "slice_type": "eMBB", "a3_margin": margin}
    for ue_id, margin in ((1, 3.0), (2, 2.0), (3, 1.0))
]
accepted, rejected, debug = controller.admit_candidates(candidates)

assert [item["ue_id"] for item in accepted] == [1, 2]
assert rejected[0]["rejection_reason"] == "direction_quota_exhausted"
assert controller.get_state()["direction_used"][(0, 1, "eMBB")] == 0

controller.commit(accepted[0])
assert controller.get_state()["direction_used"][(0, 1, "eMBB")] == 1

{
    "quota": controller.get_state()["direction_quota"][(0, 1, "eMBB")],
    "accepted_batch": [item["ue_id"] for item in accepted],
    "real_used_after_one_commit": controller.get_state()["direction_used"][(0, 1, "eMBB")],
    "rejection_reasons": debug["rejection_reasons"],
}

## 4. Exact-Direction Offset Neutralization

Only the exhausted `(source, target, slice)` offset should be neutralized.
The other negative offset from the same source remains active.

In [ ]:
fake_env = GlobalPPO3GNBEnv.__new__(GlobalPPO3GNBEnv)
fake_env.safe_admission_enabled = True
fake_env.n_gnbs = 3
fake_env.slice_types = ("eMBB", "URLLC", "mMTC")
fake_env.neighbors = {0: (1, 2), 1: (0, 2), 2: (0, 1)}
fake_env.base_env = SimpleNamespace(get_safe_admission_state=lambda: {
    "direction_quota": {(1, 0, "eMBB"): 1, (1, 2, "eMBB"): 2},
    "direction_used": {(1, 0, "eMBB"): 1, (1, 2, "eMBB"): 1},
})

offsets = np.zeros((3, 2, 3), dtype=float)
offsets[1, 0, 0] = -6.0  # g1 -> g0 eMBB exhausted
offsets[1, 1, 0] = -6.0  # g1 -> g2 eMBB still has budget

neutralized = GlobalPPO3GNBEnv._zero_quota_exhausted_offsets(fake_env, offsets)

assert neutralized[1, 0, 0] == 0.0
assert neutralized[1, 1, 0] == -6.0

{
    "before_g1_eMBB_offsets": offsets[1, :, 0].tolist(),
    "after_g1_eMBB_offsets": neutralized[1, :, 0].tolist(),
}

## 5. Real Upper-Environment Smoke

This checks the integration path: `GlobalPPO3GNBEnv.step()` creates the same
directional admission budget state from the upper action. Mobility/A3 guards
can still prevent actual handovers; this cell only verifies the budget layer.

In [ ]:
env = GlobalPPO3GNBEnv(
    seed=2,
    scenario_mode="curriculum",
    training_scenarios="jain_balance_controllable",
    scenario_selection="cycle",
    gnb_configs=CENTER_GAP_GNB_CONFIGS["medium_270m"],
    upper_window_seconds=1.0,
    local_steps_per_global=10,
    radio_substeps=20,
    terminal_reward_only=False,
    safe_admission_enabled=True,
)

try:
    env.reset(seed=2)
    action = np.zeros(env.action_space.shape, dtype=np.float32)
    action.reshape(3, 2, 3)[1, :, 0] = -0.8
    _obs, reward, _terminated, _truncated, info = env.step(action)
finally:
    env.close()

direction_quota = info["safe_admission"]["direction_quota"]
direction_used = info["safe_admission"]["direction_used"]
stats = info["safe_admission"]["stats"]

assert direction_quota[(1, 0, "eMBB")] == 5
assert direction_quota[(1, 2, "eMBB")] == 5
assert stats["rejected_target_safety"] == 0
assert stats["rejected_no_source_excess"] == 0
assert np.isfinite(reward)

{
    "direction_quota_g1_eMBB": {
        "to_g0": direction_quota[(1, 0, "eMBB")],
        "to_g2": direction_quota[(1, 2, "eMBB")],
    },
    "direction_used_g1_eMBB": {
        "to_g0": direction_used[(1, 0, "eMBB")],
        "to_g2": direction_used[(1, 2, "eMBB")],
    },
    "handover_count": info["handover_count"],
    "reward": reward,
}